# 第4章 地図で考える / Thinking with Maps
『AIに頼んで動かす Python実務データ分析』第4章の参照用ノートブックです。 / Reference notebook for Chapter 4.

`LANG` を選んで、すべてのセルを上から実行します。配布データ5つは、サポートページから自動で読み込まれます（またはファイルパネルからアップロード）。
Choose `LANG` and run all cells. The five companion data files are downloaded automatically (or upload them).

出典 / Sources: 国土数値情報（行政区域データ）国土交通省、令和2年国勢調査・令和3年経済センサス-活動調査（総務省統計局、e-Stat）、推計昼間人口メッシュデータ（はんけトケ、e-Stat の原典を加工）、国土数値情報（駅別乗降客数データ）国土交通省

In [ ]:
LANG = "ja"   # "ja" / "en"
BASE_URL = "https://raw.githubusercontent.com/rekishi-data/ai-python-data-analysis/main/data/"

In [ ]:
import os, subprocess, urllib.request, warnings; warnings.filterwarnings("ignore")
subprocess.run("pip install -q geopandas folium mapclassify", shell=True)
FILES = ["tokyo23_wards.geojson", "mesh250_population_tokyo23.csv", "mesh250_daytime_tokyo23.csv",
         "mesh500_establishments_tokyo23.csv", "stations_tokyo23_2023.csv"]
for f in FILES:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
JP = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if not os.path.exists(JP):
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True)

## メッシュコードから区画の四角形を作る / Building grid-cell polygons from mesh codes

In [ ]:
import numpy as np
def mesh_sw(code):
    c=str(code); lat=int(c[0:2])/1.5; lon=int(c[2:4])+100
    lat+=int(c[4])*5/60; lon+=int(c[5])*7.5/60
    lat+=int(c[6])*0.5/60; lon+=int(c[7])*0.75/60
    dlat,dlon=0.5/60,0.75/60
    for d in c[8:]:
        dlat/=2; dlon/=2; k=int(d)
        if k in (3,4): lat+=dlat
        if k in (2,4): lon+=dlon
    return lat,lon,dlat,dlon
def mesh_center(code):
    lat,lon,dlat,dlon=mesh_sw(code); return lat+dlat/2, lon+dlon/2
def mesh_polygon(code):
    from shapely.geometry import box
    lat,lon,dlat,dlon=mesh_sw(code); return box(lon,lat,lon+dlon,lat+dlat)

## 4-1 23区の地図を描く / Mapping the 23 wards

In [ ]:
import os, pandas as pd, numpy as np, geopandas as gpd, matplotlib
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
JP="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"; fm.fontManager.addfont(JP)
plt.rcParams.update({"font.family":fm.FontProperties(fname=JP).get_name(),"font.size":8,"savefig.dpi":300})
EN={"千代田区":"Chiyoda","中央区":"Chuo","港区":"Minato","新宿区":"Shinjuku","文京区":"Bunkyo","台東区":"Taito","墨田区":"Sumida","江東区":"Koto","品川区":"Shinagawa","目黒区":"Meguro","大田区":"Ota","世田谷区":"Setagaya","渋谷区":"Shibuya","中野区":"Nakano","杉並区":"Suginami","豊島区":"Toshima","北区":"Kita","荒川区":"Arakawa","板橋区":"Itabashi","練馬区":"Nerima","足立区":"Adachi","葛飾区":"Katsushika","江戸川区":"Edogawa"}
nm=lambda w: w if LANG=="ja" else EN[w]
w=gpd.read_file("tokyo23_wards.geojson").to_crs(6677)
p=pd.read_csv("mesh250_population_tokyo23.csv")
pop=p.groupby("ward_ja").pop_total.sum(); w["pop"]=w.ward_ja.map(pop); w["density"]=w["pop"]/w.area_km2
w.drop(columns="geometry").to_csv("ward_summary.csv",index=False)
halo=[pe.withStroke(linewidth=1.8,foreground="white")]
def labels(ax,fs=6):
    for _,r in w.iterrows():
        pt=r.geometry.representative_point(); ax.text(pt.x,pt.y,nm(r.ward_ja),ha="center",va="center",fontsize=fs,path_effects=halo)
# 4-1-1
fig,ax=plt.subplots(figsize=(4.5,4.0)); w.plot(ax=ax,facecolor="white",edgecolor="black",lw=0.6); labels(ax); ax.set_axis_off()
fig.savefig(f"{out}/fig4-1-1_wards.png",bbox_inches="tight"); plt.show()
# 4-1-2 choropleth
bins=[0,12000,15000,18000,21000,1e9]; cols=["#f0f0f0","#c8c8c8","#969696","#636363","#303030"]
lab_ja=["12,000未満","12,000〜15,000","15,000〜18,000","18,000〜21,000","21,000以上"]
lab_en=["< 12,000","12,000–15,000","15,000–18,000","18,000–21,000","≥ 21,000"]
w["cls"]=pd.cut(w.density,bins,labels=False,right=False)
fig,ax=plt.subplots(figsize=(4.5,3.4)); w.plot(ax=ax,color=[cols[c] for c in w.cls],edgecolor="white",lw=0.8)
w.boundary.plot(ax=ax,color="black",lw=0.3); labels(ax,5.5); ax.set_axis_off()
ttl="人口密度（人/km²）" if LANG=="ja" else "Population density (per km²)"
ax.legend(handles=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(cols,lab_ja if LANG=="ja" else lab_en)],title=ttl,loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.5,title_fontsize=7,frameon=False)
fig.savefig(f"{out}/fig4-1-2_density.png",bbox_inches="tight"); plt.show()
print(w[["ward_ja","pop","area_km2","density","cls"]].sort_values("density",ascending=False).round(0).to_string())

In [ ]:
import folium
wj = w.to_crs(4326)
# 背景地図：国土地理院の淡色地図（APIキー不要、出典の明示が必要）
# Basemap: GSI "pale" tiles (no API key; attribution required)
GSI_PALE = "https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png"
GSI_ATTR = '<a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank">地理院タイル</a>'
m = folium.Map(location=[35.69, 139.75], zoom_start=11, tiles=GSI_PALE, attr=GSI_ATTR)
folium.Choropleth(geo_data=wj.__geo_interface__, data=wj, columns=["ward_ja", "density"],
                  key_on="feature.properties.ward_ja", fill_color="Greys", fill_opacity=0.7, line_color="black",
                  legend_name="人口密度（人/km²）" if LANG == "ja" else "Population density (per km²)",
                  name="人口密度" if LANG == "ja" else "Population density").add_to(m)
folium.GeoJson(wj, style_function=lambda f: {"fillOpacity": 0, "weight": 0},
               tooltip=folium.GeoJsonTooltip(fields=["ward_ja", "density"], aliases=["区", "人口密度"], localize=True)).add_to(m)
folium.LayerControl().add_to(m)
m.save(f"tokyo23_density_{LANG}.html")
m

## 4-2〜4-4 区画で見る人口・昼夜・飲食店 / Population, day-night, and restaurants by grid cell

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
JP="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"; fm.fontManager.addfont(JP)
plt.rcParams.update({"font.family":fm.FontProperties(fname=JP).get_name(),"font.size":8,"savefig.dpi":300,"axes.spines.top":False,"axes.spines.right":False})
J=LANG=="ja"
EN={"千代田区":"Chiyoda","中央区":"Chuo","港区":"Minato","新宿区":"Shinjuku","文京区":"Bunkyo","台東区":"Taito","墨田区":"Sumida","江東区":"Koto","品川区":"Shinagawa","目黒区":"Meguro","大田区":"Ota","世田谷区":"Setagaya","渋谷区":"Shibuya","中野区":"Nakano","杉並区":"Suginami","豊島区":"Toshima","北区":"Kita","荒川区":"Arakawa","板橋区":"Itabashi","練馬区":"Nerima","足立区":"Adachi","葛飾区":"Katsushika","江戸川区":"Edogawa"}
STEN={"新宿":"Shinjuku","池袋":"Ikebukuro","渋谷":"Shibuya","東京":"Tokyo","北千住":"Kita-Senju","品川":"Shinagawa","高田馬場":"Takadanobaba","新橋":"Shimbashi","秋葉原":"Akihabara","目黒":"Meguro","上野":"Ueno","大手町":"Otemachi"}
nm=lambda w: w if J else EN[w]
W=gpd.read_file("tokyo23_wards.geojson").to_crs(6677)
def meshgdf(df,col="mesh_code"):
    return gpd.GeoDataFrame(df.copy(),geometry=[mesh_polygon(str(c)) for c in df[col]],crs=4326).to_crs(6677)
C5=["#f0f0f0","#c8c8c8","#969696","#636363","#252525"]
def choro(g,val,bins,labs,title,fname,na_note=None):
    g=g.copy(); g["cls"]=pd.cut(g[val],bins,labels=False,right=False)
    fig,ax=plt.subplots(figsize=(4.5,3.4))
    W.plot(ax=ax,facecolor="white",edgecolor="none")
    g[g.cls.notna()].plot(ax=ax,color=[C5[int(c)] for c in g.cls.dropna()],edgecolor="none")
    W.boundary.plot(ax=ax,color="black",lw=0.35); ax.set_axis_off()
    h=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,labs)]
    if na_note: h.append(Patch(fc="white",ec="black",lw=0.4,label=na_note))
    ax.legend(handles=h,title=title,loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.3,title_fontsize=7,frameon=False)
    fig.savefig(f"{out}/{fname}.png",bbox_inches="tight"); plt.show()
stats={}
# ---- 4-2 population mesh
p=pd.read_csv("mesh250_population_tokyo23.csv",dtype={"mesh_code":str}); gp=meshgdf(p)
choro(gp,"pop_total",[1,500,1000,1500,2000,1e9],
      ["1〜499","500〜999","1,000〜1,499","1,500〜1,999","2,000以上"] if J else ["1–499","500–999","1,000–1,499","1,500–1,999","≥ 2,000"],
      "人口（人／区画）" if J else "Population per cell","fig4-2-1_mesh_population")
stats["n_mesh"]=len(p); stats["pop_max"]=p.loc[p.pop_total.idxmax(),["mesh_code","ward_ja","pop_total"]].tolist()
stats["mesh_ge4000"]=int((p.pop_total>=4000).sum()); stats["mesh_lt100"]=int((p.pop_total<100).sum())
p["old_rate"]=p.pop_65plus/p.pop_total*100; p["fr_rate"]=p.pop_foreign/p.pop_total*100
ok=p[(p.pop_total>=100)&p.pop_65plus.notna()]
g2=meshgdf(ok)
choro(g2,"old_rate",[0,15,20,25,30,101],["15%未満","15〜20%","20〜25%","25〜30%","30%以上"] if J else ["< 15%","15–20%","20–25%","25–30%","≥ 30%"],
      "高齢化率" if J else "Share aged 65+","fig4-2-2_mesh_aging","人口100人未満・秘匿" if J else "Pop. < 100 / suppressed")
stats["old_ward"]=(p.groupby("ward_ja").pop_65plus.sum()/p.groupby("ward_ja").pop_total.sum()*100).round(1).sort_values().to_dict()
stats["old_mesh_ge30"]=int((ok.old_rate>=30).sum()); stats["old_mesh_lt15"]=int((ok.old_rate<15).sum()); stats["old_mesh_n"]=len(ok)
ok2=p[(p.pop_total>=100)&p.pop_foreign.notna()]
g3=meshgdf(ok2)
choro(g3,"fr_rate",[0,2,4,6,10,101],["2%未満","2〜4%","4〜6%","6〜10%","10%以上"] if J else ["< 2%","2–4%","4–6%","6–10%","≥ 10%"],
      "外国人の割合" if J else "Foreign residents","fig4-2-3_mesh_foreign","人口100人未満・秘匿" if J else "Pop. < 100 / suppressed")
top=ok2.sort_values("fr_rate",ascending=False).head(10)[["mesh_code","ward_ja","pop_total","pop_foreign","fr_rate","lat","lon"]]
stats["fr_top"]=top.round(4).values.tolist(); stats["fr_ge10_by_ward"]=ok2[ok2.fr_rate>=10].ward_ja.value_counts().to_dict()
stats["secret_old"]=int(p.pop_65plus.isna().sum()); stats["secret_fr"]=int(p.pop_foreign.isna().sum())
# ---- 4-3 daytime
d=pd.read_csv("mesh250_daytime_tokyo23.csv",dtype={"mesh_code":str})
wd=d.groupby("ward_ja")[["night_pop","day_pop"]].sum(); wd["ratio"]=wd.day_pop/wd.night_pop; wd=wd.sort_values("ratio")
fig,ax=plt.subplots(figsize=(4.5,3.6))
ax.barh([nm(x) for x in wd.index],wd.ratio,color=["#555555" if r>=1 else "#bbbbbb" for r in wd.ratio],edgecolor="black",lw=0.5)
ax.set_xscale("log"); ax.axvline(1,color="black",lw=0.8,ls="--")
for i,v in enumerate(wd.ratio): ax.text(v*1.05,i,f"{v:.2f}",va="center",fontsize=6.3)
ax.set_xlabel("昼夜間人口比（対数目盛）" if J else "Day-night population ratio (log scale)"); ax.tick_params(axis="y",labelsize=6.8)
ax.set_xticks([0.5,1,2,5,10,20]); ax.set_xticklabels(["0.5","1","2","5","10","20"]); ax.set_xlim(0.5,40)
fig.tight_layout(); fig.savefig(f"{out}/fig4-3-1_ward_daynight.png",bbox_inches="tight"); plt.show()
stats["ward_ratio"]=wd.ratio.round(2).to_dict(); stats["total_ratio"]=round(d.day_pop.sum()/d.night_pop.sum(),2)
d["ratio"]=np.where(d.night_pop>0,d.day_pop/d.night_pop.replace(0,np.nan),np.inf)
gd=meshgdf(d[(d.day_pop+d.night_pop)>0])
st=pd.read_csv("stations_tokyo23_2023.csv").head(10)
gs=gpd.GeoDataFrame(st,geometry=gpd.points_from_xy(st.lon,st.lat),crs=4326).to_crs(6677)
gd["cls"]=pd.cut(gd.ratio.replace(np.inf,1e9),[0,0.8,1.25,2,5,1e10],labels=False,right=False)
fig,ax=plt.subplots(figsize=(4.5,3.4)); W.plot(ax=ax,facecolor="white",edgecolor="none")
gd.plot(ax=ax,color=[C5[int(c)] for c in gd.cls],edgecolor="none"); W.boundary.plot(ax=ax,color="black",lw=0.35)
ax.scatter(gs.geometry.x,gs.geometry.y,s=gs.passengers_per_day/40000,facecolor="white",edgecolor="black",lw=0.8,zorder=5)
halo=[pe.withStroke(linewidth=1.6,foreground="white")]
for _,r in gs.head(6).iterrows(): ax.text(r.geometry.x+700,r.geometry.y+300,r.station_name if J else STEN.get(r.station_name,r.station_name),fontsize=6.5,path_effects=halo,zorder=6)
ax.set_axis_off()
labs=["0.8未満","0.8〜1.25","1.25〜2","2〜5","5以上"] if J else ["< 0.8","0.8–1.25","1.25–2","2–5","≥ 5"]
h=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,labs)]
h.append(plt.Line2D([],[],marker="o",ls="",mfc="white",mec="black",ms=6,label="乗降客数上位10駅" if J else "Top 10 stations"))
ax.legend(handles=h,title="昼夜間人口比" if J else "Day-night ratio",loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.3,title_fontsize=7,frameon=False)
fig.savefig(f"{out}/fig4-3-2_mesh_daynight.png",bbox_inches="tight"); plt.show()
stats["mesh_ratio_counts"]=gd.cls.value_counts().sort_index().to_dict(); stats["stations_top10"]=st[["station_name","passengers_per_day"]].values.tolist()
# ---- 4-4 establishments & restaurants
e=pd.read_csv("mesh500_establishments_tokyo23.csv",dtype={"mesh_code":str})
d["m500"]=d.mesh_code.str[:9]; d5=d.groupby("m500")[["night_pop","day_pop"]].sum()
p["m500"]=p.mesh_code.str[:9]; p5=p.groupby("m500")[["pop_total","pop_foreign"]].sum(min_count=1)
m=e.set_index("mesh_code").join(d5).join(p5)
ge=meshgdf(e)
choro(ge,"est_restaurant",[1,20,50,100,200,1e9],["1〜19","20〜49","50〜99","100〜199","200以上"] if J else ["1–19","20–49","50–99","100–199","≥ 200"],
      "飲食店の数（500m区画）" if J else "Restaurants per 500 m cell","fig4-4-1_restaurants")
top=m.sort_values("est_restaurant",ascending=False).head(10)
# nearest station name
stall=pd.read_csv("stations_tokyo23_2023.csv")
def nearest(lat,lon):
    dd=np.hypot((stall.lat-lat)*111,(stall.lon-lon)*91); i=dd.idxmin(); return stall.station_name[i]
top["near"]=[nearest(a,b) for a,b in zip(top.lat,top.lon)]
stats["rest_top"]=top[["ward_ja","near","est_restaurant","day_pop","night_pop"]].values.tolist()
mm=m.dropna(subset=["day_pop","night_pop"]); mm=mm[(mm.est_restaurant>0)]
stats["corr_log"]={k:round(np.corrcoef(np.log1p(mm.est_restaurant),np.log1p(mm[k]))[0,1],2) for k in ["day_pop","night_pop","pop_foreign"] if k in mm}
stats["corr_raw"]={k:round(mm[["est_restaurant",k]].corr().iloc[0,1],2) for k in ["day_pop","night_pop","pop_foreign"]}
fig,axs=plt.subplots(1,2,figsize=(4.5,2.3),sharey=True)
for ax,k,t in zip(axs,["night_pop","day_pop"],[("夜間人口（人）","Night population"),("昼間人口（人）","Day population")]):
    ax.scatter(mm[k],mm.est_restaurant,s=3,color="black",alpha=0.35,lw=0)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel(t[0] if J else t[1],fontsize=7); ax.tick_params(labelsize=6.5)
    ax.set_title(f"r = {stats['corr_log'][k]:.2f}",fontsize=7.5)
axs[0].set_ylabel("飲食店の数" if J else "Restaurants",fontsize=7)
fig.tight_layout(); fig.savefig(f"{out}/fig4-4-2_scatter.png",bbox_inches="tight"); plt.show()
stats["n500"]=len(e); stats["n_scatter"]=len(mm); stats["rest_total"]=int(e.est_restaurant.sum())
import json; print(json.dumps(stats,ensure_ascii=False,default=str)[:3000])
print("done")